In [1]:
import pandas as pd
import numpy as np
import re

#Set to true to invert mutations
INVERT = True
TAXA = 'Blattodea'
#Set to True to calculate skews for CO1 genes only
JUST_CO1 = False
if JUST_CO1:
    PATH_TO_CODONTABLE = f'../data/codontable_midori_{TAXA}_CO1.csv'
    PATH_TO_SKEW = f"../data/midori_{TAXA}_CO1_skew.csv"
else:
    PATH_TO_CODONTABLE = f'../data/codontable_midori_{TAXA}.csv'
    PATH_TO_SKEW = f"../data/midori_{TAXA}_skew.csv"

def get_skew_df(codon_table):
    codon_table = pd.read_csv(codon_table)

    codon_table = codon_table.reset_index(drop=True)
    
    ###ADD IT TO SEPARATE SUBSOCIAL CRIPTOCERCUS
    if TAXA == 'Blattodea':
        codon_table.loc[codon_table["Taxonomy"] == "['Eukaryota_2759', 'Arthropoda_6656', 'Insecta_50557', 'Blattodea_85823', 'Cryptocercidae_36982']",
                                        "Workers"] = 'Sub'
    
    ###Assigning organism type
    if TAXA == 'Blattodea':
        workers = codon_table['Workers']
    #such a mess :(
    elif TAXA == 'Diptera':
        codon_table['Organism'] = ""
        nematocera_families = ['Anisopodidae_52748','Bibionidae_52729','Cecidomyiidae_33406','Ceratopogonidae_41819','Chaoboridae_41811','Chironomidae_7149','Culicidae_7157','Keroplatidae_58254',
            'Limoniidae_43823','Mycetophilidae_29035','Psychodidae_7197','Ptychopteridae_79304','Sciaridae_7184','Simuliidae_7190','Tipulidae_41042']

        for fam in nematocera_families:
            for i in codon_table.index:
                real_fam = re.sub("['\[\]]",'',codon_table['Taxonomy'][i].split(',')[4].strip())
                if real_fam == fam:
                    codon_table.at[i, 'Organism'] = 'Nematocera'

        brachycera_families = ['Agromyzidae_127399','Anthomyiidae_30062','Asilidae_50673','Aulacigastridae_286480','Calliphoridae_7371','Chamaemyiidae_189958','Chloropidae_29032',
            'Clusiidae_286472','Conopidae_115263','Dolichopodidae_92558','Drosophilidae_7214','Dryomyzidae_169441','Empididae_92557','Ephydridae_48991',
            'Fanniidae_27471','Fergusoninidae_156410','Glossinidae_7392','Heleomyzidae_219548','Hippoboscidae_81710','Hybotidae_1446258','Lauxaniidae_189929',
            'Milichiidae_305559','Muscidae_7366','Mydidae_50677','Nemestrinidae_92615','Nycteribiidae_81707','Oestridae_7387','Opomyzidae_286476','Phoridae_36164',
            'Piophilidae_28629','Pipunculidae_43835','Platypezidae_43827','Platystomatidae_28632','Polleniidae_54279','Rhagionidae_92609','Sarcophagidae_7381',
            'Scathophagidae_43756','Sciomyzidae_169447','Sepsidae_137503','Sphaeroceridae_114620','Stratiomyidae_34687','Streblidae_81697','Syrphidae_34680',
            'Tachinidae_27474','Tephritidae_7211','Tabanidae_7205','Xylophagaidae_92613']
        for fam in brachycera_families:
            for i in codon_table.index:
                real_fam = re.sub("['\[\]]",'',codon_table['Taxonomy'][i].split(',')[4].strip())
                if real_fam == fam:
                    codon_table.at[i, 'Organism'] = 'Brachycera'
        #Removing species without set organism type
        codon_table = codon_table.drop(codon_table[codon_table['Organism'] == ""].index)
        organism = codon_table['Organism']
    elif TAXA == 'Apidae':
        codon_table['Organism'] = ""
        social_genuses = ['Bombus', 'Apis', 'Exoneurella', 'Ceratina', 'Melipona', 'Partamona', 'Euglossa', 'Tetragonula']
        nonsocial_genuses = ['Xylocopa', 'Amegilla', 'Anthophora', 'Nomada', 'Eulaema', 'Eucera', 'Epeolus']
        for genus in social_genuses:
            for i in codon_table.index:
                real_genus = codon_table['Species_name'][i].split('_')[0]
                if real_genus == genus:
                    codon_table.at[i, 'Organism'] = 'Social'
        for genus in nonsocial_genuses:
            for i in codon_table.index:
                real_genus = codon_table['Species_name'][i].split('_')[0]
                if real_genus == genus:
                    codon_table.at[i, 'Organism'] = 'Nonsocial'
        #Removing species without set organism type
        codon_table = codon_table.drop(codon_table[codon_table['Organism'] == ""].index)
        organism = codon_table['Organism']
    elif TAXA == 'Hymenoptera':
        codon_table['Organism'] = ""
        long_tongue_fams = ['Apidae_7458', 'Megachilidae_124286']

        for fam in long_tongue_fams:
            for i in codon_table.index:
                real_fam = re.sub("['\[\]]",'',codon_table['Taxonomy'][i].split(',')[4].strip())
                if real_fam == fam:
                    codon_table.at[i, 'Organism'] = 'Long tongued'

        short_tongue_fams = ['Andrenidae_48719', 'Halictidae_77572', 'Colletidae_156309']
        for fam in short_tongue_fams:
            for i in codon_table.index:
                real_fam = re.sub("['\[\]]",'',codon_table['Taxonomy'][i].split(',')[4].strip())
                if real_fam == fam:
                    codon_table.at[i, 'Organism'] = 'Short tongued'
        #Removing species without set organism type
        codon_table = codon_table.drop(codon_table[codon_table['Organism'] == ""].index)
        organism = codon_table['Organism']
    else:
        codon_table['Organism'] = ""
        organism = codon_table['Organism']
    
    neg_genes = ['ND1', 'ND4', 'ND4L', 'ND5']
    if INVERT == True:
        GAskew = []
        TCskew = []
        for i in codon_table.index:
            if codon_table['Gene_name'][i] not in neg_genes:
                GAskew.append((codon_table['neutralC'][i] - codon_table['neutralT'][i])/(codon_table['neutralC'][i] + codon_table['neutralT'][i]))
                TCskew.append((codon_table['neutralA'][i] - codon_table['neutralG'][i])/(codon_table['neutralA'][i] + codon_table['neutralG'][i]))
            else:
                GAskew.append((codon_table['neutralG'][i] - codon_table['neutralA'][i])/(codon_table['neutralG'][i] + codon_table['neutralA'][i]))
                TCskew.append((codon_table['neutralT'][i] - codon_table['neutralC'][i])/(codon_table['neutralT'][i] + codon_table['neutralC'][i]))

        IDs = codon_table['Species_name']
        genes = codon_table['Gene_name']
        
                
        skews = IDs.to_frame().merge(pd.Series(GAskew).rename('GAskew'), left_index=True, right_index=True)
        skews = skews.merge(genes.rename('Gene_name'), left_index=True, right_index=True)
        skews = skews.merge(pd.Series(TCskew).rename('TCskew'), left_index=True, right_index=True)
        
        if TAXA == 'Blattodea':
            skews = skews.merge(workers.rename('Organism'), left_index=True, right_index=True)
        else:
            skews = skews.merge(organism.rename('Organism'), left_index=True, right_index=True)
    else:
        AGskew = []
        CTskew = []
        for i in codon_table.index:
            if codon_table['Gene_name'][i] not in neg_genes:
                AGskew.append((codon_table['neutralA'][i] - codon_table['neutralG'][i])/(codon_table['neutralA'][i] + codon_table['neutralG'][i]))
                CTskew.append((codon_table['neutralC'][i] - codon_table['neutralT'][i])/(codon_table['neutralC'][i] + codon_table['neutralT'][i]))
            else:
                AGskew.append((codon_table['neutralT'][i] - codon_table['neutralC'][i])/(codon_table['neutralT'][i] + codon_table['neutralC'][i]))
                CTskew.append((codon_table['neutralG'][i] - codon_table['neutralA'][i])/(codon_table['neutralG'][i] + codon_table['neutralA'][i]))

        IDs = codon_table['Species_name']
        genes = codon_table['Gene_name']
        
                
        skews = IDs.to_frame().merge(pd.Series(AGskew).rename('AGskew'), left_index=True, right_index=True)
        skews = skews.merge(genes.rename('Gene_name'), left_index=True, right_index=True)
        skews = skews.merge(pd.Series(CTskew).rename('CTskew'), left_index=True, right_index=True)
        
        if TAXA == 'Blattodea':
            skews = skews.merge(workers.rename('Organism'), left_index=True, right_index=True)
        else:
            skews = skews.merge(organism.rename('Organism'), left_index=True, right_index=True)        
    return (skews)

derrived_skew = get_skew_df(PATH_TO_CODONTABLE)
if TAXA == 'Blattodea' or TAXA == 'Diptera' or TAXA == 'Apidae' or TAXA == 'Hymenoptera':
    derrived_skew = derrived_skew.sort_values(by=['Organism', 'Species_name'], ascending=True)
else:
    derrived_skew = derrived_skew.sort_values(by=['Species_name'], ascending=True)
if TAXA == 'Blattodea':
    derrived_skew['Organism'] = derrived_skew['Organism'].map({1.0 : 'Termites w/ workers', 0.0 : 'Termites w/o workers', "Sub" : 'Sub-social Cryptocercus'}) ### LAST ONE SEPARATES SUBSOCIAL SPECIES
    derrived_skew['Organism'].fillna('Cockroaches', inplace=True)
derrived_skew = derrived_skew.replace(
    [np.inf, -np.inf],
    np.nan
)
derrived_skew.to_csv(
    PATH_TO_SKEW,
    index=False,
    na_rep="NA"
)

#one hot encoding for cocks_phylo_stats.r
if TAXA == 'Blattodea':
    cock_terms_skew = derrived_skew
    cock_terms_skew = pd.concat([cock_terms_skew, pd.get_dummies(cock_terms_skew.Organism, dtype=int)], axis='columns')
    cock_terms_skew = cock_terms_skew.drop(columns=['Organism'])
    cock_terms_skew.to_csv('../data/midori_Blattodea_skew_onehot_encoded.csv', index=False)

C:\Users\voron\AppData\Local\Temp\ipykernel_20028\789723938.py:103: RuntimeWarning: invalid value encountered in scalar divide
  GAskew.append((codon_table['neutralC'][i] - codon_table['neutralT'][i])/(codon_table['neutralC'][i] + codon_table['neutralT'][i]))
